In [ ]:
# Campus Carpool Coordination Portal
# Beginner-friendly Python project using functions, lists, dictionaries,
# loops, conditions, exception handling, and JSON file handling.
# The three JSON files will be created automatically in the notebook's current working directory.

import json
import os
import math
import re
import tempfile
from datetime import datetime

STUDENTS_FILE = "students.json"
RIDES_FILE = "rides.json"
BOOKINGS_FILE = "bookings.json"

students = []
rides = []
bookings = []


# ------------------------------------------------------------
# FILE HANDLING
# ------------------------------------------------------------

def load_json_file(file_name):
    """Load a list of records from a JSON file, creating it if missing."""
    if not os.path.exists(file_name):
        with open(file_name, "w", encoding="utf-8") as file:
            json.dump([], file, indent=4)
        return []

    try:
        with open(file_name, "r", encoding="utf-8") as file:
            records = json.load(file)

        if not isinstance(records, list):
            raise ValueError(
                f"{file_name} must contain a JSON list. "
                "Please correct the file before running the portal again."
            )

        return records

    except json.JSONDecodeError as error:
        raise ValueError(
            f"{file_name} contains invalid JSON. "
            "Please fix the file rather than overwriting its data."
        ) from error


def validate_loaded_records(file_name, records):
    """Check required fields and data types in every loaded JSON record."""
    required_fields = {
        STUDENTS_FILE: {"student_id": str, "name": str, "email": str, "phone": str, "password": str, "role": str},
        RIDES_FILE: {"ride_id": str, "driver_id": str, "pickup": str, "destination": str, "date": str, "time": str, "total_seats": int, "available_seats": int, "fare": (int, float), "status": str},
        BOOKINGS_FILE: {"booking_id": str, "ride_id": str, "passenger_id": str, "seats_booked": int, "total_fare": (int, float), "status": str}
    }
    for index, record in enumerate(records, start=1):
        if not isinstance(record, dict):
            raise ValueError(f"{file_name}: record {index} must be a JSON object.")
        for field, expected_type in required_fields[file_name].items():
            if field not in record:
                raise ValueError(f"{file_name}: record {index} is missing '{field}'.")
            value = record[field]
            if isinstance(value, bool) or not isinstance(value, expected_type):
                raise ValueError(f"{file_name}: '{field}' in record {index} has an invalid data type.")
        if file_name == STUDENTS_FILE and record["role"] not in ("Driver", "Passenger"):
            raise ValueError(f"{file_name}: record {index} has an invalid role.")
        if file_name == RIDES_FILE:
            if record["total_seats"] <= 0 or not 0 <= record["available_seats"] <= record["total_seats"]:
                raise ValueError(f"{file_name}: record {index} has invalid seat values.")
            if not math.isfinite(record["fare"]) or record["fare"] <= 0:
                raise ValueError(f"{file_name}: record {index} has an invalid fare.")
            try:
                datetime.strptime(record["date"], "%Y-%m-%d")
                datetime.strptime(record["time"], "%H:%M")
            except ValueError as error:
                raise ValueError(f"{file_name}: record {index} has an invalid date or time.") from error
        if file_name == BOOKINGS_FILE:
            if record["seats_booked"] <= 0 or not math.isfinite(record["total_fare"]) or record["total_fare"] < 0:
                raise ValueError(f"{file_name}: record {index} has invalid booking values.")


def load_data():
    """Load and validate all portal data before using it."""
    global students, rides, bookings
    loaded_students = load_json_file(STUDENTS_FILE)
    loaded_rides = load_json_file(RIDES_FILE)
    loaded_bookings = load_json_file(BOOKINGS_FILE)
    validate_loaded_records(STUDENTS_FILE, loaded_students)
    validate_loaded_records(RIDES_FILE, loaded_rides)
    validate_loaded_records(BOOKINGS_FILE, loaded_bookings)
    students, rides, bookings = loaded_students, loaded_rides, loaded_bookings


def save_json_file(file_name, records):
    """Write to a temporary file first, then safely replace the JSON file."""
    directory = os.path.dirname(os.path.abspath(file_name))
    temporary_path = None
    try:
        with tempfile.NamedTemporaryFile(mode="w", encoding="utf-8", dir=directory, delete=False, suffix=".tmp") as file:
            json.dump(records, file, indent=4, allow_nan=False)
            temporary_path = file.name
        os.replace(temporary_path, file_name)
    except (OSError, TypeError, ValueError) as error:
        if temporary_path and os.path.exists(temporary_path):
            os.remove(temporary_path)
        raise error


def save_data():
    """Save all portal data and return True only if every file was saved."""
    try:
        save_json_file(STUDENTS_FILE, students)
        save_json_file(RIDES_FILE, rides)
        save_json_file(BOOKINGS_FILE, bookings)
        return True
    except (OSError, TypeError, ValueError) as error:
        print(f"ERROR: Could not save portal data: {error}")
        return False


# ------------------------------------------------------------
# GENERAL HELPER FUNCTIONS
# ------------------------------------------------------------

def get_next_id(prefix, records, id_field):
    """Create the next ID, such as S001, R001, or B001."""
    highest_number = 0

    for record in records:
        record_id = record.get(id_field, "")
        if record_id.startswith(prefix):
            number_part = record_id[len(prefix):]
            if number_part.isdigit():
                highest_number = max(highest_number, int(number_part))

    return f"{prefix}{highest_number + 1:03d}"


def get_non_empty_input(prompt):
    """Keep asking until the user enters something other than blank spaces."""
    while True:
        value = input(prompt).strip()
        if value:
            return value
        print("This field cannot be left empty.")


def get_positive_integer(prompt):
    """Ask for a whole number greater than zero."""
    while True:
        try:
            number = int(input(prompt))
            if number > 0:
                return number
            print("Please enter a number greater than zero.")
        except ValueError:
            print("Please enter a valid whole number.")


def get_positive_fare(prompt):
    """Ask for a finite fare greater than zero."""
    while True:
        try:
            fare = float(input(prompt))
            if math.isfinite(fare) and fare > 0:
                return round(fare, 2)
            print("Fare must be a finite positive number.")
        except ValueError:
            print("Please enter a valid amount, for example 200 or 200.50.")


def validate_student_id(value):
    return bool(re.fullmatch(r"[A-Za-z0-9-]{3,20}", value))


def validate_email(value):
    return bool(re.fullmatch(r"[A-Za-z0-9.!#$%&'*+/=?^_`{|}~-]+@[A-Za-z0-9-]+(?:\.[A-Za-z0-9-]+)+", value))


def validate_pakistani_phone(value):
    return bool(re.fullmatch(r"(?:03\d{9}|\+923\d{9})", value))


def validate_password(value):
    return len(value) >= 8 and bool(re.search(r"[A-Za-z]", value)) and bool(re.search(r"\d", value))


def get_validated_input(prompt, validator, error_message):
    while True:
        value = get_non_empty_input(prompt)
        if validator(value):
            return value
        print(error_message)


def get_validated_student_id(prompt):
    return get_validated_input(prompt, validate_student_id, "Student ID must contain 3-20 letters, numbers, or hyphens.")


def get_validated_email(prompt):
    return get_validated_input(prompt, validate_email, "Enter a valid email address, for example student@gmail.com.")


def get_validated_phone(prompt):
    return get_validated_input(prompt, validate_pakistani_phone, "Enter an 11-digit Pakistani number starting with 03, or use +92 format.")


def get_validated_password(prompt):
    return get_validated_input(prompt, validate_password, "Password must be at least 8 characters and contain at least one letter and one number.")


def find_student(student_id):
    """Return a student dictionary using an exact, case-sensitive student ID."""
    for student in students:
        if student.get("student_id", "") == student_id.strip():
            return student
    return None


def find_ride(ride_id):
    """Return a ride dictionary using an exact, case-sensitive ride ID."""
    for ride in rides:
        if ride.get("ride_id", "") == ride_id.strip():
            return ride
    return None


def find_booking(booking_id):
    """Return a booking dictionary using an exact, case-sensitive booking ID."""
    for booking in bookings:
        if booking.get("booking_id", "") == booking_id.strip():
            return booking
    return None


def get_booked_seat_count(ride_id):
    """Count seats in confirmed bookings for a particular ride."""
    seat_count = 0

    for booking in bookings:
        if (booking.get("ride_id", "") == ride_id
                and booking.get("status") == "Confirmed"):
            seat_count += booking.get("seats_booked", 0)

    return seat_count


def display_ride(ride, show_phone=False):
    """Display ride details. Driver phone is hidden unless access is allowed."""
    driver = find_student(ride.get("driver_id", ""))
    driver_name = driver.get("name", "Unknown") if driver else "Unknown"

    print("-" * 48)
    print(f"Ride ID:          {ride.get('ride_id')}")
    print(f"Driver:           {driver_name}")
    print(f"Pickup:           {ride.get('pickup')}")
    print(f"Destination:      {ride.get('destination')}")
    print(f"Date:             {ride.get('date')}")
    print(f"Departure time:   {ride.get('time')}")
    print(f"Available seats:  {ride.get('available_seats')}")
    print(f"Fare per seat:    Rs. {ride.get('fare'):.2f}")
    print(f"Status:           {ride.get('status')}")

    if show_phone and driver:
        print(f"Driver phone:     {driver.get('phone', 'Not provided')}")
    print("-" * 48)


def validate_future_datetime(date_text, time_text, show_message=True):
    """Validate date and time together so a departure earlier today is rejected."""
    try:
        departure = datetime.strptime(f"{date_text} {time_text}", "%Y-%m-%d %H:%M")
        if departure <= datetime.now():
            if show_message:
                print("The ride's departure date and time must be in the future.")
            return False
        return True
    except ValueError:
        if show_message:
            print("Use date YYYY-MM-DD and time HH:MM (24-hour format).")
        return False


def validate_future_date(date_text):
    try:
        datetime.strptime(date_text, "%Y-%m-%d")
        return True
    except ValueError:
        print("Please use the date format YYYY-MM-DD, for example 2026-12-05.")
        return False


def validate_time(time_text):
    try:
        datetime.strptime(time_text, "%H:%M")
        return True
    except ValueError:
        print("Please use 24-hour time format, for example 08:30 or 14:15.")
        return False


def is_future_ride(ride):
    try:
        departure = datetime.strptime(f"{ride.get('date', '')} {ride.get('time', '')}", "%Y-%m-%d %H:%M")
        return departure > datetime.now()
    except (TypeError, ValueError):
        return False


def confirm_change_to_booked_ride(ride):
    """Warn the driver before changing details of a ride with confirmed bookings."""
    has_bookings = any(
        booking.get("ride_id", "") == ride.get("ride_id", "")
        and booking.get("status") == "Confirmed" for booking in bookings
    )
    if not has_bookings:
        return True
    print("WARNING: Passengers have already booked this ride.")
    answer = input("Changing these details may affect passengers. Continue? (yes/no): ").strip().casefold()
    return answer == "yes"


# ------------------------------------------------------------
# STUDENT REGISTRATION AND LOGIN
# ------------------------------------------------------------

def register_student():
    """Register a new student as either a driver or a passenger."""
    print("\n========== STUDENT REGISTRATION ==========")

    student_id = get_validated_student_id("Enter student ID: ")

    if find_student(student_id):
        print("This student ID is already registered.")
        return

    name = get_non_empty_input("Enter full name: ")
    email = get_validated_email("Enter email address: ")
    if any(existing.get("email", "").casefold() == email.casefold() for existing in students):
        print("This email address is already registered.")
        return
    phone = get_validated_phone("Enter phone number: ")
    password = get_validated_password("Create a password: ")

    print("\nSelect your role:")
    print("1. Driver")
    print("2. Passenger")

    while True:
        role_choice = input("Choose an option (1-2): ").strip()
        if role_choice == "1":
            role = "Driver"
            break
        elif role_choice == "2":
            role = "Passenger"
            break
        else:
            print("Please select 1 for Driver or 2 for Passenger.")

    student = {
        "student_id": student_id,
        "name": name,
        "email": email,
        "phone": phone,
        "password": password,
        "role": role
    }

    students.append(student)
    if save_data():
        print(f"\nRegistration successful! Your role is {role}.")
        print("You can now log in using your student ID and password.")
    else:
        students.remove(student)
        print("Registration was not completed because the data could not be saved.")


def login_student():
    """Check a student's login details and open the correct menu."""
    print("\n================ STUDENT LOGIN ================")

    student_id = get_non_empty_input("Enter student ID: ")
    password = get_non_empty_input("Enter password: ")

    student = find_student(student_id)

    if student is None or student.get("password") != password:
        print("Incorrect student ID or password.")
        return

    print(f"\nWelcome, {student.get('name')}!")

    if student.get("role") == "Driver":
        driver_menu(student)
    elif student.get("role") == "Passenger":
        passenger_menu(student)
    else:
        print("This account has an invalid role. Please contact the project administrator.")


# ------------------------------------------------------------
# RIDE MANAGEMENT - DRIVER FUNCTIONS
# ------------------------------------------------------------

def post_ride(driver):
    """Allow a logged-in driver to publish a one-time ride."""
    print("\n================ POST A NEW RIDE ================")

    pickup = get_non_empty_input("Enter pickup location: ")
    destination = get_non_empty_input("Enter destination: ")

    if pickup.casefold() == destination.casefold():
        print("Pickup and destination cannot be the same.")
        return

    while True:
        ride_date = get_non_empty_input("Enter date (YYYY-MM-DD): ")
        if not validate_future_date(ride_date):
            continue
        ride_time = get_non_empty_input("Enter departure time (HH:MM, 24-hour): ")
        if validate_time(ride_time) and validate_future_datetime(ride_date, ride_time):
            break

    total_seats = get_positive_integer("Enter available passenger seats: ")
    fare = get_positive_fare("Enter fare per passenger (Rs.): ")

    ride_id = get_next_id("R", rides, "ride_id")

    ride = {
        "ride_id": ride_id,
        "driver_id": driver["student_id"],
        "pickup": pickup,
        "destination": destination,
        "date": ride_date,
        "time": ride_time,
        "total_seats": total_seats,
        "available_seats": total_seats,
        "fare": fare,
        "status": "Active"
    }

    rides.append(ride)
    if save_data():
        print(f"\nRide posted successfully. Your ride ID is {ride_id}.")
    else:
        rides.remove(ride)
        print("Ride was not posted because the data could not be saved.")


def view_my_rides(driver):
    """Show all rides posted by the logged-in driver."""
    print("\n================ MY RIDES ================")
    found = False

    for ride in rides:
        if ride.get("driver_id") == driver["student_id"]:
            display_ride(ride)
            found = True

    if not found:
        print("You have not posted any rides yet.")


def update_ride(driver):
    """Allow a driver to update details of their own active ride."""
    import copy
    print("\n================ UPDATE RIDE ================")
    ride_id = get_non_empty_input("Enter the ride ID you want to update: ")
    ride = find_ride(ride_id)

    if ride is None:
        print("Ride not found.")
        return

    if ride.get("driver_id") != driver["student_id"]:
        print("You can only update rides that you posted.")
        return

    if ride.get("status") != "Active":
        print("Only active rides can be updated.")
        return

    original_ride = copy.deepcopy(ride)
    print("\nWhat would you like to update?")
    print("1. Pickup location")
    print("2. Destination")
    print("3. Date")
    print("4. Departure time")
    print("5. Fare per passenger")
    print("6. Total passenger seats")
    print("7. Return to driver menu")

    choice = input("Choose an option (1-7): ").strip()

    if choice in {"1", "2", "3", "4", "5", "6"}:
        if not confirm_change_to_booked_ride(ride):
            print("Update cancelled. No ride details were changed.")
            return

    if choice == "1":
        new_pickup = get_non_empty_input("Enter new pickup location: ")
        if new_pickup.casefold() == ride["destination"].casefold():
            print("Pickup and destination cannot be the same.")
            return
        ride["pickup"] = new_pickup

    elif choice == "2":
        new_destination = get_non_empty_input("Enter new destination: ")
        if new_destination.casefold() == ride["pickup"].casefold():
            print("Pickup and destination cannot be the same.")
            return
        ride["destination"] = new_destination

    elif choice == "3":
        while True:
            new_date = get_non_empty_input("Enter new date (YYYY-MM-DD): ")
            if (validate_future_date(new_date)
                    and validate_future_datetime(new_date, ride["time"])):
                break
        ride["date"] = new_date

    elif choice == "4":
        while True:
            new_time = get_non_empty_input("Enter new time (HH:MM): ")
            if validate_time(new_time) and validate_future_datetime(ride["date"], new_time):
                break
        ride["time"] = new_time

    elif choice == "5":
        ride["fare"] = get_positive_fare("Enter new fare per passenger (Rs.): ")
        print("The new fare will apply to future bookings. Existing bookings keep their original fare.")

    elif choice == "6":
        booked_seats = get_booked_seat_count(ride["ride_id"])
        minimum_seats = booked_seats

        while True:
            new_total = get_positive_integer("Enter the new total passenger seats: ")
            if new_total >= minimum_seats:
                ride["total_seats"] = new_total
                ride["available_seats"] = new_total - booked_seats
                break
            print(f"Total seats cannot be less than the {booked_seats} seats already booked.")

    elif choice == "7":
        return

    else:
        print("Invalid choice.")
        return

    if save_data():
        print("Ride details updated successfully.")
    else:
        ride.clear()
        ride.update(original_ride)
        print("Ride update was not completed because saving failed.")


def cancel_ride(driver):
    """Cancel a driver's ride and cancel all confirmed bookings for it."""
    print("\n================ CANCEL RIDE ================")
    ride_id = get_non_empty_input("Enter the ride ID to cancel: ")
    ride = find_ride(ride_id)

    if ride is None:
        print("Ride not found.")
        return

    if ride.get("driver_id") != driver["student_id"]:
        print("You can only cancel rides that you posted.")
        return

    if ride.get("status") != "Active":
        print("This ride is already cancelled.")
        return

    confirmation = input("Are you sure you want to cancel this ride? (yes/no): ").strip().lower()

    if confirmation != "yes":
        print("Ride cancellation stopped.")
        return

    import copy
    original_ride = copy.deepcopy(ride)
    original_bookings = copy.deepcopy(bookings)
    ride["status"] = "Cancelled"

    for booking in bookings:
        if (booking.get("ride_id", "") == ride["ride_id"]
                and booking.get("status") == "Confirmed"):
            booking["status"] = "Cancelled"

    if save_data():
        print("Ride cancelled. Its confirmed bookings have also been cancelled.")
    else:
        ride.clear()
        ride.update(original_ride)
        bookings[:] = original_bookings
        print("Ride cancellation was not completed because saving failed.")


def view_passenger_bookings(driver):
    """Show confirmed passenger bookings for the driver's own rides."""
    print("\n================ PASSENGER BOOKINGS ================")
    found = False

    for ride in rides:
        if ride.get("driver_id") == driver["student_id"]:
            for booking in bookings:
                if (booking.get("ride_id", "") == ride.get("ride_id", "")
                        and booking.get("status") == "Confirmed"):
                    passenger = find_student(booking.get("passenger_id", ""))

                    print("-" * 45)
                    print(f"Ride ID:       {ride.get('ride_id')}")
                    print(f"Booking ID:    {booking.get('booking_id')}")
                    print(f"Passenger:     {passenger.get('name') if passenger else 'Unknown'}")
                    print(f"Phone:         {passenger.get('phone') if passenger else 'Not available'}")
                    print(f"Seats booked:  {booking.get('seats_booked')}")
                    print("-" * 45)
                    found = True

    if not found:
        print("There are no confirmed passenger bookings for your rides.")


# ------------------------------------------------------------
# RIDE SEARCH AND DISPLAY - PASSENGER FUNCTIONS
# ------------------------------------------------------------

def view_available_rides():
    """Display active rides that still have at least one available seat."""
    print("\n================ AVAILABLE RIDES ================")
    found = False

    for ride in rides:
        if (ride.get("status") == "Active"
                and ride.get("available_seats", 0) > 0
                and is_future_ride(ride)):
            display_ride(ride, show_phone=False)
            found = True

    if not found:
        print("No rides are currently available.")


def search_rides():
    """Search for active rides using exact pickup and destination locations."""
    print("\n================ SEARCH RIDES ================")
    pickup = get_non_empty_input("Enter your pickup location: ")
    destination = get_non_empty_input("Enter your destination: ")

    found = False

    for ride in rides:
        if (ride.get("pickup", "").casefold() == pickup.casefold()
                and ride.get("destination", "").casefold() == destination.casefold()
                and ride.get("status") == "Active"
                and ride.get("available_seats", 0) > 0
                and is_future_ride(ride)):
            display_ride(ride, show_phone=False)
            found = True

    if not found:
        print("No available rides match those locations.")


# ------------------------------------------------------------
# BOOKING MANAGEMENT - PASSENGER FUNCTIONS
# ------------------------------------------------------------

def book_ride(passenger):
    """Allow a passenger to book seats on an available ride."""
    print("\n================ BOOK A RIDE ================")
    ride_id = get_non_empty_input("Enter the ride ID you want to book: ")
    ride = find_ride(ride_id)

    if ride is None:
        print("Ride not found.")
        return

    if ride.get("driver_id") == passenger["student_id"]:
        print("You cannot book your own ride.")
        return

    if ride.get("status") != "Active":
        print("This ride is not active.")
        return

    if not is_future_ride(ride):
        print("This ride has already departed or its departure time has passed.")
        return

    ride_id = ride["ride_id"]  # Always use the ID stored in the ride record.

    if ride.get("available_seats", 0) <= 0:
        print("Sorry, this ride is full.")
        return

    # Only one active booking per passenger for the same ride.
    for booking in bookings:
        if (booking.get("ride_id", "") == ride_id
                and booking.get("passenger_id") == passenger["student_id"]
                and booking.get("status") == "Confirmed"):
            print("You already have a confirmed booking for this ride.")
            return

    print(f"Available seats: {ride['available_seats']}")

    while True:
        seats_requested = get_positive_integer("How many seats would you like to book? ")
        if seats_requested <= ride["available_seats"]:
            break
        print("There are not enough seats available. Please enter a smaller number.")

    total_fare = round(seats_requested * ride["fare"], 2)
    booking_id = get_next_id("B", bookings, "booking_id")

    booking = {
        "booking_id": booking_id,
        "ride_id": ride_id,
        "passenger_id": passenger["student_id"],
        "seats_booked": seats_requested,
        "total_fare": total_fare,
        "status": "Confirmed"
    }

    old_available_seats = ride["available_seats"]
    bookings.append(booking)
    ride["available_seats"] -= seats_requested

    if save_data():
        print("\nBooking confirmed!")
        print(f"Booking ID: {booking_id}")
        print(f"Seats booked: {seats_requested}")
        print(f"Total fare: Rs. {total_fare:.2f}")
        print("The driver's phone number is available in your confirmed booking details.")
    else:
        bookings.remove(booking)
        ride["available_seats"] = old_available_seats
        print("Booking was not confirmed because the data could not be saved.")


def view_my_bookings(passenger):
    """Show a passenger's bookings and reveal driver contact for confirmed ones."""
    print("\n================ MY BOOKINGS ================")
    found = False

    for booking in bookings:
        if booking.get("passenger_id") == passenger["student_id"]:
            ride = find_ride(booking.get("ride_id", ""))

            print("-" * 48)
            print(f"Booking ID:   {booking.get('booking_id')}")
            print(f"Ride ID:      {booking.get('ride_id')}")
            print(f"Seats:        {booking.get('seats_booked')}")
            print(f"Total fare:   Rs. {booking.get('total_fare', 0):.2f}")
            print(f"Status:       {booking.get('status')}")

            if booking.get("status") == "Confirmed" and ride:
                driver = find_student(ride.get("driver_id", ""))
                if driver:
                    print(f"Driver:       {driver.get('name')}")
                    print(f"Driver phone: {driver.get('phone')}")
                    print(f"Pickup:       {ride.get('pickup')}")
                    print(f"Destination:  {ride.get('destination')}")
                    print(f"Date:         {ride.get('date')}")
                    print(f"Time:         {ride.get('time')}")
            print("-" * 48)
            found = True

    if not found:
        print("You have not made any bookings yet.")


def cancel_booking(passenger):
    """Cancel a passenger's confirmed booking and return the seats to the ride."""
    print("\n================ CANCEL BOOKING ================")
    booking_id = get_non_empty_input("Enter the booking ID to cancel: ")
    booking = find_booking(booking_id)

    if booking is None:
        print("Booking not found.")
        return

    if booking.get("passenger_id") != passenger["student_id"]:
        print("You can only cancel your own bookings.")
        return

    if booking.get("status") != "Confirmed":
        print("This booking is not currently confirmed.")
        return

    ride = find_ride(booking.get("ride_id", ""))

    if ride is None:
        print("The associated ride could not be found. Please contact the project administrator.")
        return

    confirmation = input("Are you sure you want to cancel this booking? (yes/no): ").strip().lower()

    if confirmation != "yes":
        print("Booking cancellation stopped.")
        return

    import copy
    original_booking = copy.deepcopy(booking)
    original_available_seats = ride.get("available_seats")

    # Restore seats only when cancelling a confirmed booking on an active ride.
    if ride.get("status") == "Active":
        ride["available_seats"] += booking["seats_booked"]
        ride["available_seats"] = min(ride["available_seats"], ride["total_seats"])

    booking["status"] = "Cancelled"
    if save_data():
        print("Your booking has been cancelled successfully.")
        print("The seats have been made available again if the ride is still active.")
    else:
        booking.clear()
        booking.update(original_booking)
        if original_available_seats is not None:
            ride["available_seats"] = original_available_seats
        print("Booking cancellation was not completed because saving failed.")


# ------------------------------------------------------------
# DRIVER AND PASSENGER MENUS
# ------------------------------------------------------------

def driver_menu(driver):
    """Display the menu and options available to a driver."""
    while True:
        print("\n======================================")
        print("              DRIVER MENU")
        print("======================================")
        print(f"Welcome, {driver['name']}!")
        print("1. Post a New Ride")
        print("2. View My Rides")
        print("3. Update Ride Details")
        print("4. Cancel a Ride")
        print("5. View Passenger Bookings")
        print("6. Logout")

        choice = input("Choose an option (1-6): ").strip()

        if choice == "1":
            post_ride(driver)
        elif choice == "2":
            view_my_rides(driver)
        elif choice == "3":
            update_ride(driver)
        elif choice == "4":
            cancel_ride(driver)
        elif choice == "5":
            view_passenger_bookings(driver)
        elif choice == "6":
            print("You have logged out.")
            break
        else:
            print("Invalid choice. Please select a number from 1 to 6.")


def passenger_menu(passenger):
    """Display the menu and options available to a passenger."""
    while True:
        print("\n======================================")
        print("            PASSENGER MENU")
        print("======================================")
        print(f"Welcome, {passenger['name']}!")
        print("1. View Available Rides")
        print("2. Search Rides")
        print("3. Book a Ride")
        print("4. View My Bookings")
        print("5. Cancel a Booking")
        print("6. Logout")

        choice = input("Choose an option (1-6): ").strip()

        if choice == "1":
            view_available_rides()
        elif choice == "2":
            search_rides()
        elif choice == "3":
            book_ride(passenger)
        elif choice == "4":
            view_my_bookings(passenger)
        elif choice == "5":
            cancel_booking(passenger)
        elif choice == "6":
            print("You have logged out.")
            break
        else:
            print("Invalid choice. Please select a number from 1 to 6.")


# ------------------------------------------------------------
# MAIN MENU
# ------------------------------------------------------------

def main():
    """Start the portal and keep showing the main menu until the user exits."""
    try:
        load_data()
    except (ValueError, OSError) as error:
        print(f"\nThe portal could not load its data: {error}")
        print("Please check the JSON files and run the program again.")
        return

    print("\nData loaded successfully.")

    while True:
        print("\n==============================================")
        print("       CAMPUS CARPOOL COORDINATION PORTAL")
        print("==============================================")
        print("1. Register as a Student")
        print("2. Login")
        print("3. View Available Rides")
        print("4. Exit")

        choice = input("Choose an option (1-4): ").strip()

        if choice == "1":
            register_student()
        elif choice == "2":
            login_student()
        elif choice == "3":
            view_available_rides()
        elif choice == "4":
            if save_data():
                print("Thank you for using the Campus Carpool Portal!")
                break
            print("The portal could not save its data. Please try exiting again.")
        else:
            print("Invalid choice. Please select a number from 1 to 4.")


# Run the portal.
main()



Data loaded successfully.

       CAMPUS CARPOOL COORDINATION PORTAL
1. Register as a Student
2. Login
3. View Available Rides
4. Exit
Choose an option (1-4): 1

========== STUDENT REGISTRATION ==========
Enter student ID: 24l-5048
Enter full name: ali
Enter email address: l245048
Enter a valid email address, for example student@gmail.com.
Enter email address: l245048@lhr.nu.edu.pk
Enter phone number: 03344467831
Create a password: ali12345

Select your role:
1. Driver
2. Passenger
Choose an option (1-2): 2

Registration successful! Your role is Passenger.
You can now log in using your student ID and password.

       CAMPUS CARPOOL COORDINATION PORTAL
1. Register as a Student
2. Login
3. View Available Rides
4. Exit
Choose an option (1-4): 3

================ AVAILABLE RIDES ================
------------------------------------------------
Ride ID:          R001
Driver:           malaika
Pickup:           johar town
Destination:      fast nuces
Date:             2026-11-15
Departure 